# User Dataset

Build user-level representations from the interaction records.


In [9]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "data" / "processed").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

INPUT_PATH = PROCESSED_DIR / "interaction_records.csv"
OUTPUT_HISTORY = PROCESSED_DIR / "user_interaction_history.csv"
OUTPUT_USERS = PROCESSED_DIR / "user_dataset.csv"

print("Input:", INPUT_PATH)
print("History output:", OUTPUT_HISTORY)
print("User dataset:", OUTPUT_USERS)


Input: f:\annuspeaks.com\recommendation-system\data\processed\interaction_records.csv
History output: f:\annuspeaks.com\recommendation-system\data\processed\user_interaction_history.csv
User dataset: f:\annuspeaks.com\recommendation-system\data\processed\user_dataset.csv


## User Interaction History

The canonical interaction history remains in row-level form. Records are sorted chronologically and written incrementally to avoid constructing a large nested Python object in memory.


In [10]:
# Build user interaction history

if OUTPUT_HISTORY.exists():
    OUTPUT_HISTORY.unlink()

first_write = True

for chunk in pd.read_csv(
    INPUT_PATH,
    usecols=["user_id", "item_id", "interaction_type", "weight", "timestamp"],
    chunksize=250_000,
):
    chunk = chunk.sort_values(["user_id", "timestamp"])

    chunk.to_csv(
        OUTPUT_HISTORY,
        mode="w" if first_write else "a",
        header=first_write,
        index=False,
    )

    first_write = False

print("User interaction history created:")
print(OUTPUT_HISTORY)
print("Size:", f"{OUTPUT_HISTORY.stat().st_size / (1024**2):.2f} MB")


User interaction history created:
f:\annuspeaks.com\recommendation-system\data\processed\user_interaction_history.csv
Size: 95.04 MB


## Aggregate User Activity & Create User-Level Statistics


In [11]:
# Aggregate user activity and create exact user-level statistics.
# Counts and timestamps are accumulated chunk-by-chunk.

interaction_count = {}
total_weight = {}
first_timestamp = {}
last_timestamp = {}

# Exact user-item pairs ensure unique_products is not over-counted
# when a user appears in multiple chunks.
unique_pairs = set()

for chunk in pd.read_csv(
    INPUT_PATH,
    usecols=["user_id", "item_id", "weight", "timestamp"],
    chunksize=250_000,
):
    grouped = chunk.groupby("user_id").agg(
        interaction_count=("item_id", "size"),
        total_preference_weight=("weight", "sum"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )

    for user_id, row in grouped.iterrows():
        interaction_count[user_id] = (
            interaction_count.get(user_id, 0)
            + int(row["interaction_count"])
        )
        total_weight[user_id] = (
            total_weight.get(user_id, 0.0)
            + float(row["total_preference_weight"])
        )
        first_timestamp[user_id] = min(
            first_timestamp.get(user_id, row["first_timestamp"]),
            row["first_timestamp"],
        )
        last_timestamp[user_id] = max(
            last_timestamp.get(user_id, row["last_timestamp"]),
            row["last_timestamp"],
        )

    unique_pairs.update(zip(chunk["user_id"], chunk["item_id"]))

unique_product_counts = {}

for user_id, item_id in unique_pairs:
    unique_product_counts[user_id] = (
        unique_product_counts.get(user_id, 0) + 1
    )

user_stats = pd.DataFrame({
    "user_id": list(interaction_count.keys()),
    "interaction_count": list(interaction_count.values()),
    "unique_products": [
        unique_product_counts[user_id]
        for user_id in interaction_count
    ],
    "total_preference_weight": [
        total_weight[user_id]
        for user_id in interaction_count
    ],
    "first_timestamp": [
        first_timestamp[user_id]
        for user_id in interaction_count
    ],
    "last_timestamp": [
        last_timestamp[user_id]
        for user_id in interaction_count
    ],
})

print("Users:", f"{len(user_stats):,}")
display(user_stats.head())


Users: 1,407,580


,user_id,interaction_count,unique_products,total_preference_weight,first_timestamp,last_timestamp
0,17,1,1,1.0,1433211837727,1433211837727
1,23,3,2,3.0,1433650323043,1434077771188
2,28,1,1,1.0,1433580742154,1433580742154
3,52,1,1,1.0,1433198245927,1433198245927
4,62,2,1,2.0,1434173353423,1434218761600


## Recency / Frequency Signals


In [12]:
# Use the latest observed interaction as the dataset reference point.
# This avoids using the current calendar date.

reference_timestamp = user_stats["last_timestamp"].max()

user_stats["recency"] = (
    reference_timestamp - user_stats["last_timestamp"]
)

user_stats["frequency"] = user_stats["interaction_count"]

print("Reference timestamp:", reference_timestamp)
display(
    user_stats[
        ["interaction_count", "recency", "frequency"]
    ].head()
)


Reference timestamp: 1442545187788


,interaction_count,recency,frequency
0,1,9333350061,1
1,3,8467416600,3
2,1,8964445634,1
3,1,9346941861,1
4,2,8326426188,2


## Active / Inactive Users

Initial rule: a user is active when their latest observed interaction falls within the latest 30-day window of the dataset.


In [13]:
ACTIVE_WINDOW_MS = 30 * 24 * 60 * 60 * 1000

user_stats["active"] = (
    user_stats["recency"] <= ACTIVE_WINDOW_MS
)

user_stats["activity_status"] = user_stats["active"].map({
    True: "active",
    False: "inactive",
})

print(user_stats["activity_status"].value_counts())


activity_status
inactive    1109135
active       298445
Name: count, dtype: int64


In [14]:
# Save canonical user-level dataset

user_stats.to_csv(
    OUTPUT_USERS,
    index=False
)

print("Saved:", OUTPUT_USERS)
print("Rows:", f"{len(user_stats):,}")
display(user_stats.head())


Saved: f:\annuspeaks.com\recommendation-system\data\processed\user_dataset.csv
Rows: 1,407,580


,user_id,interaction_count,unique_products,total_preference_weight,first_timestamp,last_timestamp,recency,frequency,active,activity_status
0,17,1,1,1.0,1433211837727,1433211837727,9333350061,1,False,inactive
1,23,3,2,3.0,1433650323043,1434077771188,8467416600,3,False,inactive
2,28,1,1,1.0,1433580742154,1433580742154,8964445634,1,False,inactive
3,52,1,1,1.0,1433198245927,1433198245927,9346941861,1,False,inactive
4,62,2,1,2.0,1434173353423,1434218761600,8326426188,2,False,inactive


In [15]:
# Final Phase 2.2 validation

required_columns = [
    "user_id",
    "interaction_count",
    "unique_products",
    "total_preference_weight",
    "first_timestamp",
    "last_timestamp",
    "recency",
    "frequency",
    "active",
    "activity_status",
]

missing_columns = [
    column for column in required_columns
    if column not in user_stats.columns
]

print("Missing required columns:", missing_columns)
print("Users:", f"{user_stats['user_id'].nunique():,}")
print("Active users:", f"{user_stats['active'].sum():,}")
print("Inactive users:", f"{(~user_stats['active']).sum():,}")

assert not missing_columns
assert user_stats["user_id"].notna().all()
assert (user_stats["interaction_count"] > 0).all()
assert (user_stats["unique_products"] > 0).all()


Missing required columns: []
Users: 1,407,580
Active users: 298,445
Inactive users: 1,109,135


## Completion

The user dataset contains interaction history, aggregated activity, user-level statistics, recency/frequency signals, and active/inactive status.
